# COMP0178 — Query Processing Worksheet

Practice notebook based on the worksheet text you provided. It covers the first two questions only; the supplied text stops at Question 2(a)(ii).

**How to use this notebook:** read the definitions, try each part before opening its answer check, and note any assumptions. The estimates use the lecture’s simplifying assumption that attribute values are uniformly distributed unless stated otherwise.


## Statistics and notation

- `nTuples(R)`: number of tuples (rows) in relation `R`.
- `bFactor(R)`: number of tuples that fit in one disk block.
- `nBlocks(R)`: number of disk blocks occupied by `R`, approximately `nTuples(R) / bFactor(R)` (round up if calculating an integer number of blocks).
- `nDistinct_A(R)`: number of distinct values of attribute `A` in `R`.
- `min_A(R)` and `max_A(R)`: smallest and largest values of `A` in `R`.
- `SCA(R)`: selection cardinality, the estimated number of tuples from `R` that satisfy the condition on `A`.

Keep two things separate: **result size** is the number of rows returned; **disk cost** is the number of blocks read.


## Question 1 — Estimate the result size of a selection

### 1(a) Inequality condition `A > c`

If `c` lies between `min_A(R)` and `max_A(R)`, and values are approximately uniform, estimate the fraction of values above `c` from the range. Multiply that fraction by `nTuples(R)` to estimate the output rows.

Write your estimate in terms of the worksheet statistics:

```text
SCA(R, A > c) ≈ __________________________________________
```

**Check edge cases:** if `c < min_A(R)`, almost/all rows qualify; if `c >= max_A(R)`, none qualify (ignoring special boundary/discrete-value details).

**My working:**  
_Write the formula and any assumptions here._

<details><summary>Answer check for 1(a)</summary>

For `min_A(R) ≤ c < max_A(R)`, the uniform-range estimate is:

```text
SCA(R, A > c) ≈ nTuples(R) × (max_A(R) − c) / (max_A(R) − min_A(R))
```

This estimates the proportion of the attribute range above `c`. It is approximate: it assumes values are spread uniformly across the range. If `c < min_A(R)`, estimate all `nTuples(R)` rows; if `c ≥ max_A(R)`, estimate 0.
</details>


### 1(b) Equality to any value in `{C1, ..., Cn}`

For one equality value, the lecture estimates about `nTuples(R) / nDistinct_A(R)` matching rows under uniformity. For a set of distinct values, multiply the one-value estimate by the number of values.

```text
SCA(R, A ∈ {C1, ..., Cn}) ≈ _______________________________
```

The estimate cannot exceed the number of tuples in `R`.

**My working:**  
_Write the expression and state the uniform-distribution assumption._

<details><summary>Answer check for 1(b)</summary>

```text
SCA(R, A ∈ {C1, ..., Cn}) ≈ min(nTuples(R), n × nTuples(R) / nDistinct_A(R))
```

Here the set is assumed to contain `n` distinct values that occur in the relation. If some requested values do not occur, or the data is skewed, the estimate may be too high or low.
</details>


## Question 2 — Estimate disk accesses for a selection

### 2(a) Linear search on an unordered file with no index

A linear search reads blocks sequentially because there is no ordering or index to guide it.

#### 2(a)(i) Equality condition on a unique attribute

For an equality lookup on a unique attribute, at most one tuple matches. If the matching tuple is equally likely to occur anywhere in the file, a successful search finds it after reading about half the blocks on average.

```text
Average successful-search cost ≈ ___________________________
Worst-case or unsuccessful-search cost ≈ ____________________
```

**My working:**  
_Explain why the average successful lookup can stop early._

<details><summary>Answer check for 2(a)(i)</summary>

```text
Average successful-search cost ≈ nBlocks(R) / 2
Worst-case or unsuccessful-search cost ≈ nBlocks(R)
```

This is an average-case estimate for a successful search, assuming the target is equally likely to be anywhere in the unordered file. If the target is absent, or lies in the last block, the scan reads the whole file.
</details>


#### 2(a)(ii) Equality on a non-unique attribute or an inequality condition

For these conditions, the query may need to find multiple matching tuples. The search cannot stop after the first match; it must check the rest of the unordered file.

```text
Disk-access cost ≈ __________________________________________
```

**My working:**  
_Why must the scan continue to the end?_ 

<details><summary>Answer check for 2(a)(ii)</summary>

```text
Disk-access cost ≈ nBlocks(R)
```

A non-unique equality can have more matching tuples later in the file, and an inequality can match many tuples. With no index or ordering, the scan must inspect all blocks to be sure it has found every result.
</details>


## Small numerical check

Use this example to check that the formulas make sense:

```text
nTuples(R) = 1,200
nBlocks(R) = 60
nDistinct_A(R) = 12
min_A(R) = 0
max_A(R) = 120
```

1. Estimate rows for `A = 5`.
2. Estimate rows for `A ∈ {10, 20, 30}`.
3. Estimate rows for `A > 90`.
4. Estimate average successful disk accesses for unique equality using a linear search.
5. Estimate disk accesses for non-unique equality using a linear search.

**My working:**  
_Show each formula substitution._

<details><summary>Answer check</summary>

1. `1,200 / 12 = 100` rows.
2. `3 × 100 = 300` rows.
3. Range proportion above 90 is `(120 − 90) / (120 − 0) = 1/4`, so `1,200 × 1/4 = 300` rows.
4. `60 / 2 = 30` blocks on average for a successful unique lookup; up to 60 in the worst/absent case.
5. 60 blocks, because the full scan must find all matching rows.
</details>


## Summary

- Equality selectivity under a uniform assumption: `nTuples(R) / nDistinct_A(R)`.
- Inequality selectivity: estimate the fraction of the `min`–`max` range that satisfies the condition, then multiply by `nTuples(R)`.
- Set membership: multiply the one-value equality estimate by the number of distinct requested values, capped by `nTuples(R)`.
- Linear scan for a unique equality: about half the blocks on average if a match is present; up to all blocks in the worst case.
- Linear scan when all matches must be found: all `nBlocks(R)` blocks.
